Should take results from model-metrics.py for graphing and other testing in the form of a csv




Consider including WandB for logging and cloud saving.

No manual required. 

In [1]:
#imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from PIL import Image
import textwrap

# Set a clean visual style for our charts
sns.set_theme(style="whitegrid")

### Inputs

In [ ]:

#inputs

# --- File Paths ---
# Replace the filenames with the exact CSVs generated in your metrics-results folder
csv_path_model1 = "../results/metrics-results/gemma_COCO_val_20260424_102829.csv"
csv_path_model2 = "../results/metrics-results/molmo_COCO_val_TIMESTAMP.csv" # Leave as a dummy string if Model 2 isn't ready yet

# Path to original images (assuming coco-dataset is one level up from notebooks)
IMAGES_DIR = "../data/coco-dataset/val2017"



In [3]:
#data analysis 

def print_summary_stats(df, model_name):
    print(f"--- Summary Statistics for {model_name} ---")
    # Calculate mean, variance, median, min, max
    stats = df[['CIDEr_Score', 'SPICE_Score']].agg(['mean', 'var', 'median', 'min', 'max']).round(4)
    print(stats)
    print("\n")

# Load Model 1
df_m1 = pd.read_csv(csv_path_model1)
df_m1['Model'] = 'Gemma 4B'
print_summary_stats(df_m1, "Gemma 4B")

# Attempt to load Model 2 (fails gracefully if not run yet)
try:
    df_m2 = pd.read_csv(csv_path_model2)
    df_m2['Model'] = 'Molmo 2 4B'
    print_summary_stats(df_m2, "Molmo 2 4B")
    
    # Combine both into one DataFrame for easy graphing
    df_all = pd.concat([df_m1, df_m2], ignore_index=True)
except FileNotFoundError:
    print("Model 2 CSV not found. Proceeding with only Model 1 data.")
    df_all = df_m1.copy()


FileNotFoundError: [Errno 2] No such file or directory: '../results/metrics-results/gemma_COCO_val_TIMESTAMP.csv'

In [ ]:
#graphs
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Histogram of CIDEr
sns.histplot(data=df_all, x='CIDEr_Score', hue='Model', kde=True, ax=axes[0], alpha=0.6)
axes[0].set_title('CIDEr Score Distribution')
axes[0].set_xlabel('CIDEr Score')

# 2. Histogram of SPICE
sns.histplot(data=df_all, x='SPICE_Score', hue='Model', kde=True, ax=axes[1], alpha=0.6)
axes[1].set_title('SPICE Score Distribution')
axes[1].set_xlabel('SPICE Score')

# 3. Scatter Plot: CIDEr vs SPICE
sns.scatterplot(data=df_all, x='CIDEr_Score', y='SPICE_Score', hue='Model', alpha=0.5, ax=axes[2])
axes[2].set_title('CIDEr vs. SPICE Correlation')
axes[2].set_xlabel('CIDEr Score')
axes[2].set_ylabel('SPICE Score')

plt.tight_layout()
plt.show()

In [ ]:
#image sampler 

def view_samples(df, num_samples=3, sort_by=None, ascending=True):
    """
    sort_by: 'CIDEr_Score' or 'SPICE_Score' (views lowest scores if ascending=True)
    Leave sort_by=None for purely random samples.
    """
    if sort_by:
        sample_df = df.sort_values(by=sort_by, ascending=ascending).head(num_samples)
        order = "Worst" if ascending else "Best"
        print(f"--- Viewing {num_samples} {order} Samples based on {sort_by} ---\n")
    else:
        sample_df = df.sample(num_samples)
        print(f"--- Viewing {num_samples} Random Samples ---\n")
        
    for _, row in sample_df.iterrows():
        # Handle COCO ID formatting (e.g., 456865 -> 000000456865.jpg)
        img_id_str = str(row['Image_ID'])
        if not img_id_str.endswith('.jpg'):
            img_id_str = img_id_str.zfill(12) + '.jpg'
            
        img_path = os.path.join(IMAGES_DIR, img_id_str)
        
        # Display Image
        try:
            img = Image.open(img_path)
            plt.figure(figsize=(5, 3))
            plt.imshow(img)
            plt.axis('off')
            plt.show()
        except FileNotFoundError:
            print(f"[Image not found at {img_path}]")
        
        # Display Metrics & Text
        print(f"Model: {row['Model']} | Image: {img_id_str}")
        print(f"Scores -> CIDEr: {row['CIDEr_Score']:.4f} | SPICE: {row['SPICE_Score']:.4f}")
        
        wrapped_caption = textwrap.fill(row['Generated_Caption'], width=80)
        print(f"Generated Caption:\n{wrapped_caption}")
        print("-" * 80 + "\n")

# Run Examples:
# view_samples(df_all, num_samples=3) # Random
view_samples(df_all, num_samples=3, sort_by='SPICE_Score', ascending=True) # The 3 worst SPICE performers